# Stage 9a: Error analysis
Uses the saved `eval_finetuned_r16_lr2e-4.json` and `eval_base_r16_lr2e-4.json` plus the validation tweets.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q datasets pandas
import json, re, numpy as np, pandas as pd
from collections import Counter
from datasets import load_dataset

DRIVE = "/content/drive/MyDrive/finsent"
ft = json.load(open(f"{DRIVE}/eval_finetuned_r16_lr2e-4.json"))
bs = json.load(open(f"{DRIVE}/eval_base_r16_lr2e-4.json"))
val = load_dataset("zeroshot/twitter-financial-news-sentiment")["validation"]
names = ["Bearish", "Bullish", "Neutral"]

df = pd.DataFrame({"text": val["text"], "true": val["label"], "ft": ft["preds"], "base": bs["preds"]})
assert len(df) == 2388 and len(ft["preds"]) == 2388
print("ft acc", (df.true == df.ft).mean().round(4), "| base acc", (df.true == df.base).mean().round(4))

Mounted at /content/drive


README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

ft acc 0.902 | base acc 0.6059


In [2]:
# 1) Error types after fine-tuning
df["ft_err"] = df.true != df.ft
et = (df[df.ft_err].groupby(["true", "ft"]).size().reset_index(name="count"))
et["error"] = et.true.map(lambda i: names[i]) + " -> " + et.ft.map(lambda i: names[i])
print(et.sort_values("count", ascending=False)[["error", "count"]].to_string(index=False))

# 2) What fine-tuning fixed vs broke (same tweets, base vs fine-tuned)
fixed  = ((df.base != df.true) & (df.ft == df.true)).sum()
broken = ((df.base == df.true) & (df.ft != df.true)).sum()
both_wrong = ((df.base != df.true) & (df.ft != df.true)).sum()
print(f"\nfixed by fine-tuning: {fixed} | broken by fine-tuning: {broken} | wrong in both: {both_wrong}")

             error  count
Bullish -> Neutral    128
Bearish -> Neutral     57
Neutral -> Bearish     35
Neutral -> Bullish     13
Bullish -> Bearish      1

fixed by fine-tuning: 858 | broken by fine-tuning: 151 | wrong in both: 83


In [3]:
# 3) Read the actual mistakes (15 random per type, seed 42). READ THESE and note patterns.
pd.set_option("display.max_colwidth", 200)
for (t, p) in [(1, 2), (0, 2), (2, 0), (2, 1)]:
    sub = df[(df.true == t) & (df.ft == p)]
    print(f"\n=== {names[t]} -> {names[p]} ({len(sub)} tweets) ===")
    for tx in sub.sample(min(15, len(sub)), random_state=42).text:
        print(" -", tx)


=== Bullish -> Neutral (128 tweets) ===
 - $TEVA - Teva in talks with DoJ to end generics probes - Bloomberg https://t.co/a47OmArrQY
 - Madison Square Garden Q2 EPS $3.93 vs. $3.42
 - Publicis Signals Worst May Be Over for Struggling Ad Group
 - Equinor Boosts Climate Targets as Oil Output Hits Record
 - $PACD taking some. Risky but chart looks perfect for a bounce. I like this as a sympathy play off today's BORR move… https://t.co/yL5AlslU3R
 - BPH Energy : Settlement of Litigation #BPHEnergy #Stock #MarketScreener https://t.co/DD5IfT4zgo https://t.co/sZfkzFWMcP
 - China is in the process of waiving retaliatory tariffs on imports of U.S. pork and soy by domestic companies, a pro… https://t.co/08mZU9TrBX
 - $TOPS Wow..... Morgan Stanley 13G 6.7%
 - RECAP 11/22 TRUMPSAYS:
China deal is coming along very well

Live Breaking trading news
https://t.co/yChRCoaR8v
 - BP expects to keep Azeri ACG oil output stable as Shah Deniz gas powers ahead https://t.co/dcNqW5BxVN https://t.co/a0y3sr6K0u

In [4]:
# 4) Simple word patterns: words more common in Bullish->Neutral errors than in correctly-found Bullish tweets
def words(s): return re.findall(r"[a-z\$][a-z0-9\$']+", s.lower())
def contrast(err_mask, ok_mask, min_count=5, top=15):
    e, o = Counter(), Counter()
    for s in df[err_mask].text: e.update(set(words(s)))
    for s in df[ok_mask].text:  o.update(set(words(s)))
    ne, no = max(err_mask.sum(), 1), max(ok_mask.sum(), 1)
    rows = [(w, e[w], o[w], np.log(((e[w]+1)/(ne+2)) / ((o[w]+1)/(no+2)))) for w in e if e[w] >= min_count]
    return pd.DataFrame(rows, columns=["word", "in_errors", "in_correct", "log_ratio"]).sort_values("log_ratio", ascending=False).head(top)

print("Bullish->Neutral errors vs correct Bullish:")
print(contrast((df.true == 1) & (df.ft == 2), (df.true == 1) & (df.ft == 1)).round(2).to_string(index=False))
print("\nBearish->Neutral errors vs correct Bearish:")
print(contrast((df.true == 0) & (df.ft == 2), (df.true == 0) & (df.ft == 0)).round(2).to_string(index=False))

# 5) Does length matter?
df["n_words"] = df.text.str.split().str.len()
print("\nmean words | correct:", df[~df.ft_err].n_words.mean().round(1), "| errors:", df[df.ft_err].n_words.mean().round(1))
df[df.ft_err].to_csv(f"{DRIVE}/ft_errors.csv", index=False)
print("saved ft_errors.csv")

Bullish->Neutral errors vs correct Bullish:
    word  in_errors  in_correct  log_ratio
  report          5           1       2.08
      be          6           3       1.54
     has          9           5       1.50
business          6           6       0.98
   sales          7           8       0.87
  record          7           9       0.76
  growth         10          13       0.74
      is         15          20       0.71
    over          5           7       0.70
     oil          8          11       0.70
    gain          5           8       0.58
    more          7          11       0.58
 economy          9          15       0.51
 quarter          6          11       0.45
     for         18          32       0.43

Bearish->Neutral errors vs correct Bearish:
  word  in_errors  in_correct  log_ratio
  will          5           4       1.78
 china          5           9       1.09
market          5          13       0.75
    in         11          35       0.50
    by          7 